# Quickstart

Read a real Bruker TopSpin proton acquisition from the public SpectroChemPy example data, then turn its free induction decay (FID) into a spectrum. The example uses the same reference acquisition as the NMR processing gallery.

## Read and inspect the FID

The FID records the complex signal over time after excitation. Display the dataset to see its shape and time coordinate.

In [ ]:
import numpy as np
import spectrochempy as scp

fid = scp.nmr.read("nmrdata/bruker/tests/nmr/topspin_1d/1/fid")
fid

Inspect a few acquisition metadata fields. The dwell time is the interval between acquired samples.

In [ ]:
acquisition = fid.meta.acquisition
{
    "Nucleus": acquisition["nucleus"],
    "Observation frequency (MHz)": acquisition["observation_frequency_mhz"],
    "Dwell time (s)": acquisition["dwell_time_s"],
}

In [ ]:
_ = fid.plot()

## Process and plot the spectrum

Apply exponential apodization with 2 Hz line broadening, zero-fill to 32,768 points, then Fourier-transform and phase the result. The spectrum is a separate dataset with a chemical-shift axis in ppm. This call uses the SpectroChemPy processing workflow. Use an explicit TopSpin replay request when you want to apply a recorded recipe.

In [ ]:
original_data = fid.data.copy()
original_time = fid.x.data.copy()
original_history = list(fid.history)

spectrum = scp.nmr.Experiment(fid).process(
    apodization="em", lb=2.0, size=32768
)
spectrum

In [ ]:
_ = spectrum.plot(xlim=(-2.0, 12.0))

## Compare the histories

The FID keeps its import history. The spectrum records the processing steps after the import. Check that processing preserved the FID samples, time grid, and history.

In [ ]:
assert spectrum is not fid
np.testing.assert_array_equal(fid.data, original_data)
np.testing.assert_array_equal(fid.x.data, original_time)
assert fid.history == original_history

for name, dataset in [("FID", fid), ("Spectrum", spectrum)]:
    print(name)
    for entry in dataset.history:
        print(f"  {entry}")